**First section**

Reading data from `data_all_sites_aggregated.xlsx`

In [ ]:
# Libraries
import copy

import numpy as np
import pandas as pd
import pcntoolkit
from sklearn.model_selection import train_test_split
from pcntoolkit import NormData
from pathlib import Path

# Suppress some annoying warnings and logs
import warnings
import logging

pymc_logger = logging.getLogger("pymc")

pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
# pcntoolkit.util.output.Output.set_show_messages(False)

In [ ]:
#Global values
DATA_DIR = Path("data/")

In [ ]:
data_file = DATA_DIR / "data_all_sites_aggregated2.xlsx"
data = pd.read_excel(data_file)

**Data selection**

Selecting relevant clinical categories
- baseline scans
- available pet data
- Only healthy controls (HC) and major depressive disorder (MDD) diagnoses

Further, creating extra column for proper stratification of site and sex for test train split.

In [ ]:
# Stratification column for later train/test split
# strata = data["site"].astype(str) + "_" + data["sex"].astype(str)
# data = pd.concat([data, strata.rename("strata")], axis=1)

# Adding id column to ensure uniqueness of subject ids
# id column has unique id for baseline session and rescan session for same participant.
# Does not effect analysis so is not changesd
# data.insert(0, "id", range(1, len(data) + 1))
# data.to_excel("data_all_sites_aggregated2.xlsx", index = False)


# Only baseline sessions and available pet data
data = data[data["session"] == "ses-baseline"]
data = data[data["pet_avail"] == 1]

# Only healthy control group and major depression disorder (MDD) group
data = data[data["diagnosis"].isin(["HC", "MDD"])]
# print(data["strata"].value_counts())


`response_vars` is a list of variables of interest

All chosen variables have data available from NRU. There are more ROIs available in the dataset, but there is no data available from NRU.

**Filter out based on values**

1. Impossible values
   - negative values
   - zeros
2. Extreme outliers (z>5)

Kim et al. (2006) shows high test-restest reliability.

Lower for frontal cortical areas. But okay. Will filter values less than 0.05.

Take up in discussion

In [ ]:
import numpy as np
from scipy.stats import zscore

data2 = data.copy()
exclude_from_zscore_mask = data2["diagnosis"].eq("MDD")

rois = ['brain_stem', 'left_amygdala', 'left_caudate',
       'left_thalamus', 'right_amygdala', 'right_caudate', 'right_thalamus',
       'ctx_lh_rostralanteriorcingulate', 'left_putamen', 'right_putamen',
       'ctx_rh_rostralanteriorcingulate', 'ctx_rh_rostralmiddlefrontal',
       'ctx_lh_rostralmiddlefrontal']

data2 = data.copy()
exclude_from_zscore_mask = data2["diagnosis"].eq("MDD")

# remove impossible values or insignificant
nonpositive_mask = data2[rois] <= 0.05
too_large_mask = data2[rois] >= 10.0

for roi in rois:
    bad_mask = nonpositive_mask[roi]
    bad_mask2 = too_large_mask[roi]

    if bad_mask.any():
        print(f"\n{roi} | removed nonpositive {bad_mask.sum()} values:")
        print(data2.loc[bad_mask, ["site", "diagnosis", roi]])

    if bad_mask2.any():
        print(f"\n{roi} | removed too large {bad_mask2.sum()} values:")
        print(data2.loc[bad_mask2, ["site", "diagnosis", roi]])

data2[rois] = data2[rois].mask(nonpositive_mask)
data2[rois] = data2[rois].mask(too_large_mask)

# calculate site-wise z-score and remove extreme outliers
z = pd.DataFrame(index=data2.index, columns=rois, dtype=float)

for roi in rois:
    z[roi] = data2[data2["diagnosis"] == "HC"].groupby("site")[roi].transform(
        lambda x: np.abs(zscore(x, nan_policy="omit"))
    )

z_mask = (z > 5) & ~exclude_from_zscore_mask.to_numpy()[:, None]

for roi in rois:
    removed_mask = z_mask[roi]
    if removed_mask.any():
        print(f"\n{roi} | removed site-wise z>5 ({removed_mask.sum()} values):")
        print(data2.loc[removed_mask, ["site", "diagnosis", roi]])

data2[rois] = data2[rois].mask(z_mask)


**Re-binning**

Re-bin response variables based on new number of missing values 

**Split data**

Split data into relevant groups for analysis.

In [ ]:
# reading ids lists
HC_train_ids = []
with open("group_ids/HC_train_ids.txt", "r", encoding="utf-8") as f:
    for line in f:
        HC_train_ids.append(int(line))

HC_test_ids = []
with open("group_ids/HC_test_ids.txt", "r", encoding="utf-8") as f:
    for line in f:
        HC_test_ids.append(int(line))

MDD_ids = []
with open("group_ids/MDD_ids.txt", "r", encoding="utf-8") as f:
    for line in f:
        MDD_ids.append(int(line))

print(len(HC_train_ids))
print(len(HC_test_ids))
print(len(MDD_ids))


In [ ]:
## Split data into relevant categories.
data_HC_train = data2[data2["id"].isin(HC_train_ids)]
data_HC_test = data2[data2["id"].isin(HC_test_ids)]
data_MDD = data2[data2["id"].isin(MDD_ids)]

# df shape sanity check
print("HC - Training data: ", data_HC_train.shape)
print("HC - Test data: ", data_HC_test.shape)
print("MDD data: ", data_MDD.shape)

**Create NormData objects**

Create NormData objects using the data splits just defined.

These will be used as input for the normative models.

Removing na's using NormData builtin functions

In [ ]:
# Creating NormData object
# Setting covariates and batch effects
covariates = ["age"]
batch_effects = ["site"]

## structure: norm_dataset["<group>"]["<missing values bin>"]
# group : HC_train, HC_test, MDD
# missing values bins : "na_<number of na's>"
norm_dataset = {"HC_train": {},
                "HC_test": {},
                "MDD": {}
                }

for roi in rois:
        # val.append("participant_id")
        norm_dataset["HC_train"][roi] = NormData.from_dataframe(
            name = f"HC_train_{roi}", dataframe=data_HC_train, 
            covariates=covariates, batch_effects=batch_effects, 
            response_vars=[roi], remove_Nan=True, subject_ids="id"
        )

        norm_dataset["HC_test"][roi] = NormData.from_dataframe(
            name = f"HC_test_{roi}", dataframe=data_HC_test, 
            covariates=covariates, batch_effects=batch_effects, 
            response_vars=[roi], remove_Nan=True, subject_ids="id"
        )

        norm_dataset["MDD"][roi] = NormData.from_dataframe(
            name = f"data_MDD_{roi}", dataframe=data_MDD, 
            covariates=covariates, batch_effects=batch_effects, 
            response_vars=[roi], remove_Nan=True, subject_ids="id"
        )


In [ ]:
import matplotlib.pyplot as plt
from pcntoolkit import (
    HBR,
    BsplineBasisFunction,
    NormativeM  odel,
    NormalLikelihood,
    SHASHbLikelihood,
    make_prior,
    plot_centiles,
    plot_centiles_advanced,
    plot_qq,
    plot_ridge,
)

In [ ]:
# Setting up model priors 
# Hierarchical Bayesian linear regression with Normal Likelihood

mu = make_prior(
    linear = True,
    slope = make_prior(dist_name = "Normal", dist_params=(0.0, 1.0)),
    intercept = make_prior(
        random = True,
        mu=make_prior(dist_name = "Normal", dist_params = (0.0, 1.0)),
        sigma = make_prior(
                        dist_name = "Normal", 
                        dist_params= (0.0, 1.0), 
                        mapping="softplus", 
                        mapping_params=(0.0, 0.3, 0.1)
                        )
    ),
    basis_function = BsplineBasisFunction(basis_column=0, nknots=5, degree=3)
)

sigma = make_prior(
    random = True,
    dist_name = "Normal",
    mu = make_prior(
                    dist_name = "Normal", 
                    dist_params = (0.0, 1.0)
                    ),
    sigma = make_prior(
                    dist_name = "Normal", 
                    dist_params = (0.0, 1.0), 
                    mapping="softplus", 
                    mapping_params = (0.2, 1.0, 0.1)
                    ),
    mapping = "softplus",
    mapping_params = (0.2, 1.0, 0.1)
)


likelihood = NormalLikelihood(mu, sigma)

template_hbr = HBR(
    name = "template",
    cores = 16,
    progressbar=True,
    draws = 1000,
    tune = 5000,
    chains = 6,
    nuts_sampler="nutpie",
    likelihood=likelihood
)



In [ ]:
# Creating directory for normative model results
nmodel_results_dir = Path("nmodel_results")
nmodel_results_dir.mkdir(exist_ok=True)

data_version_dir = nmodel_results_dir / "agg_data"
data_version_dir.mkdir(exist_ok=True)

In [ ]:
from pathlib import Path


cdasb_model_dirs = {roi: data_version_dir / f"model_results_{roi}" # Creating Path object for each directory
                    for roi in rois}
for dir in cdasb_model_dirs.values():
    dir.mkdir(parents=True, exist_ok=True)

for key, val in cdasb_model_dirs.items():
    print(key, val)

trained_models = {}

for roi in rois:
    trained_models[roi] = NormativeModel(
        template_regression_model=template_hbr,
        savemodel=True,
        evaluate_model=True,
        saveresults=True,
        saveplots=True,
        save_dir=str(cdasb_model_dirs[roi]),
        inscaler="standardize",
        outscaler="standardize"
    )

In [ ]:
for key, model in trained_models.items():
    print(cdasb_model_dirs[key])
    print(model.save_dir)

In [ ]:
# Fit models

for key, model in trained_models.items():
    dir = Path(str(model.save_dir) + "/model" + "/normative_model.json")
    if dir.exists():
        print(f"Loading model: {key}")
        NormativeModel.load(path=str(model.save_dir), into=model)
        # model.predict(norm_dataset["HC_train"][key])
    else:
        print(f"Fitting model: {key}")
        model.fit(norm_dataset["HC_train"][key])



In [ ]:
# Predict on test and MDD set
for key, model in trained_models.items():
    model.predict(norm_dataset["HC_test"][key])
    model.predict(norm_dataset["MDD"][key])

## Save all NormData objects to file

In [ ]:
print(data_version_dir)

In [ ]:
groups = list(norm_dataset.keys())
print(groups)

for group in groups:
    for roi in rois:
        dir = data_version_dir / f"model_results_{roi}" / "data"
        # dir = Path(f"model_results_{roi}/data")
        dir.mkdir(exist_ok=True)
        dataframe = norm_dataset[group][roi].to_dataframe()
        file = str(dir) + "/" + group + ".csv"
        # print(file)
        print("saved:", file)
        dataframe.to_csv(file)
        





## Centiles plot

1. training data
2. test data
3. MDD data

Show harmonized results vs unharmonized.

Configure models and save all evaluate properly later

In [ ]:
print(len(norm_dataset["HC_train"]["na_0"]))

In [ ]:
import importlib
importlib.reload(pcntoolkit.util.plotter)
from pcntoolkit import plot_centiles_advanced
print(plot_centiles_advanced.__module__)

In [ ]:
for key, val in trained_models.items():
    print(key, val)

In [ ]:

for key, model in trained_models.items():
    dir = data_version_dir / "train_centiles"
    dir.mkdir(exist_ok=True)
    plot_centiles_advanced(
        model = model,
        scatter_data= norm_dataset["HC_train"][key],
        centiles=[0.05, 0.25,0.5, 0.75, 0.95],
        batch_effects={"site":data["site"].unique()},
        show_other_data=True,
        harmonize_data=True,
        save_dir = str(dir)
    )

    dir = data_version_dir / "test_centiles"
    dir.mkdir(exist_ok=True)
    plot_centiles_advanced(
        model = model,
        scatter_data= norm_dataset["HC_test"][key],
        centiles=[0.05, 0.25,0.5, 0.75, 0.95],
        batch_effects={"site":data["site"].unique()},
        show_other_data=True,
        harmonize_data=True,
        save_dir = str(dir)
    )

    dir = data_version_dir / "mdd_centiles"
    dir.mkdir(exist_ok=True)
    plot_centiles_advanced(
        model = model,
        scatter_data= norm_dataset["MDD"][key],
        centiles=[0.05, 0.25,0.5, 0.75, 0.95],
        batch_effects={"site":data["site"].unique()},
        show_other_data=True,
        harmonize_data=True,
        save_dir = str(dir)
    )



In [ ]:
# plotting unharmonized
for key, model in trained_models.items():
    dir = data_version_dir / "train_centiles"
    dir.mkdir(exist_ok=True)
    plot_centiles_advanced(
        model = model,
        scatter_data= norm_dataset["HC_train"][key],
        centiles=[0.05, 0.25,0.5, 0.75, 0.95],
        batch_effects={"site":data["site"].unique()},
        show_other_data=True,
        harmonize_data=False,
        save_dir = str(dir)
    )

    dir = data_version_dir / "test_centiles"
    dir.mkdir(exist_ok=True)
    plot_centiles_advanced(
        model = model,
        scatter_data= norm_dataset["HC_test"][key],
        centiles=[0.05, 0.25,0.5, 0.75, 0.95],
        batch_effects={"site":data["site"].unique()},
        show_other_data=True,
        harmonize_data=False,
        save_dir = str(dir)
    )

    dir = data_version_dir / "mdd_centiles"
    dir.mkdir(exist_ok=True)
    plot_centiles_advanced(
        model = model,
        scatter_data= norm_dataset["MDD"][key],
        centiles=[0.05, 0.25,0.5, 0.75, 0.95],
        batch_effects={"site":data["site"].unique()},
        show_other_data=True,
        harmonize_data=False,
        save_dir = str(dir)
    )



## Processing of deviation scores

Please see `postprocessing.ipynb` for analysis of deviation scores